In [1]:
from pathlib import Path
from math import pi
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

candidates = [Path.cwd(), Path.cwd() / 'Privacy-Scenario', Path.home() / 'Desktop' / 'COORDINATE-2.0' / 'Privacy-Scenario']
SCENARIO_ROOT = next((path.resolve() for path in candidates if (path / 'run_privacy_experiments.py').exists()), None)
if SCENARIO_ROOT is None:
    raise FileNotFoundError('Privacy-Scenario/run_privacy_experiments.py not found.')
ROOT = SCENARIO_ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SCENARIO_ROOT))
import run_privacy_experiments as scenario

PLOTS_DIR = SCENARIO_ROOT / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
summary_path = scenario.RESULTS_ROOT / 'privacy_summary.csv'
runs_path = scenario.RESULTS_ROOT / 'privacy_runs.csv'
raw_logs_root = scenario.RESULTS_ROOT / 'raw_logs'
if not summary_path.exists() or not runs_path.exists():
    raise FileNotFoundError('Run the full privacy campaign before generating plots.')
summary = pd.read_csv(summary_path)
runs = pd.read_csv(runs_path)
runs = runs[runs['status'] == 'ok'].copy()

technique_order = ['none', 'encryption', 'obfuscation', 'combined']
technique_labels = {'none': 'Baseline', 'encryption': 'Encryption', 'obfuscation': 'Obfuscation', 'combined': 'Combined'}
arch_order = ['CE', 'SD', 'FD']
colors = {'none': '#C9C9C9', 'encryption': '#A8C5E5', 'obfuscation': '#B8E0D2', 'combined': '#E8B4C8'}
line_colors = {'CE': '#8DA0CB', 'SD': '#FC8D62', 'FD': '#66C2A5'}

summary['privacy_resistance'] = 1.0 - pd.to_numeric(summary['avg_adversary_success_rate'], errors='coerce')
summary['privacy_resistance'] = summary['privacy_resistance'].fillna(0.0)
summary['distance_privacy'] = pd.to_numeric(summary['avg_expected_distance_error'], errors='coerce').fillna(0.0)
radar_metrics = [
    ('avg_latency_s', 'Latency', 'min'),
    ('avg_throughput_tasks_per_hour', 'Throughput', 'max'),
    ('avg_success_rate', 'Success\nRate', 'max'),
    ('avg_expiration_rate', 'Expiration\nRate', 'min'),
    ('privacy_resistance', 'Adversary\nResistance', 'max'),
    ('distance_privacy', 'Distance\nPrivacy', 'max'),
]
ranges = {}
for metric, _, _ in radar_metrics:
    values = pd.to_numeric(summary[metric], errors='coerce')
    ranges[metric] = (values.min(), values.max())

def radar_score(value, metric, direction):
    minimum, maximum = ranges[metric]
    if pd.isna(value):
        return 0.0
    if maximum == minimum:
        return 1.0
    score = (float(value) - minimum) / (maximum - minimum)
    return 1.0 - score if direction == 'min' else score

for arch in arch_order:
    metrics = radar_metrics if arch != 'FD' else radar_metrics[:4]
    angles = [2 * pi * index / len(metrics) for index in range(len(metrics))]
    closed_angles = angles + angles[:1]
    fig, ax = plt.subplots(figsize=(5.4, 5.0), subplot_kw={'polar': True})
    subset = summary[summary['architecture_label'] == arch]
    for technique in technique_order:
        row = subset[subset['technique'] == technique]
        values = []
        for metric, _, direction in metrics:
            value = pd.to_numeric(row[metric], errors='coerce').mean()
            values.append(radar_score(value, metric, direction))
        closed_values = values + values[:1]
        ax.plot(closed_angles, closed_values, color=colors[technique], linewidth=2.0, label=technique_labels[technique])
        ax.fill(closed_angles, closed_values, color=colors[technique], alpha=0.22)
    ax.set_xticks(angles)
    ax.set_xticklabels([label for _, label, _ in metrics], fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels([])
    ax.grid(color='#D6D6D6', linewidth=0.6)
    ax.set_title(f'{arch} privacy-performance trade-off', pad=24)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
    fig.tight_layout()
    filename = f'radar_{arch.lower()}_by_technique.pdf'
    fig.savefig(PLOTS_DIR / filename, bbox_inches='tight')
    plt.close(fig)
    display(IFrame(src=f'plots/{filename}', width='100%', height=470))

time_points = list(range(300, int(scenario.BASE_CONFIG['SIM_TIME']) + 1, 300))
curve_rows = []
for run in runs.itertuples(index=False):
    log_path = raw_logs_root / run.run_id / 'simulation_log.csv'
    if not log_path.exists():
        raise FileNotFoundError(f'Missing temporal log: {log_path}')
    events = pd.read_csv(log_path)
    created = events[events['event'] == 'create'][['itemId', 'simTime']].rename(columns={'simTime': 'created_at'})
    delivered = events[events['event'] == 'deliver'][['itemId', 'simTime']].rename(columns={'simTime': 'delivered_at'})
    delivered = delivered.merge(created, on='itemId', how='inner')
    delivered['latency'] = delivered['delivered_at'] - delivered['created_at']
    for sim_time in time_points:
        completed = delivered[delivered['delivered_at'] <= sim_time]
        curve_rows.append({
            'technique': run.technique,
            'architecture_label': run.architecture_label,
            'sim_time': sim_time,
            'throughput': len(completed) * 3600.0 / sim_time,
            'latency': completed['latency'].mean(),
        })
curves = pd.DataFrame(curve_rows).groupby(['technique', 'architecture_label', 'sim_time'], as_index=False).mean(numeric_only=True)

def smooth_curve(x_values, y_values, points_per_segment=24):
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    slopes = np.empty_like(y)
    slopes[0] = (y[1] - y[0]) / (x[1] - x[0])
    slopes[-1] = (y[-1] - y[-2]) / (x[-1] - x[-2])
    slopes[1:-1] = (y[2:] - y[:-2]) / (x[2:] - x[:-2])
    smooth_x = []
    smooth_y = []
    for index in range(len(x) - 1):
        step = x[index + 1] - x[index]
        t = np.linspace(0.0, 1.0, points_per_segment, endpoint=False)
        smooth_x.extend(x[index] + t * step)
        smooth_y.extend((2 * t**3 - 3 * t**2 + 1) * y[index] + (t**3 - 2 * t**2 + t) * step * slopes[index] + (-2 * t**3 + 3 * t**2) * y[index + 1] + (t**3 - t**2) * step * slopes[index + 1])
    return np.append(smooth_x, x[-1]), np.append(smooth_y, y[-1])

line_specs = [
    ('throughput', 'Mean Cumulative Throughput (tasks/hour)', 'throughput_over_time_by_mechanism.pdf'),
    ('latency', 'Mean Cumulative Latency (sec.)', 'latency_over_time_by_mechanism.pdf'),
]
for metric, ylabel, filename in line_specs:
    fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, squeeze=False)
    for index, technique in enumerate(technique_order):
        ax = axes[index // 2][index % 2]
        subset = curves[curves['technique'] == technique]
        for arch in arch_order:
            line = subset[subset['architecture_label'] == arch].sort_values('sim_time')
            measured_x = line['sim_time'].to_numpy() / 60.0
            measured_y = line[metric].to_numpy()
            valid = np.isfinite(measured_y)
            smooth_x, smooth_y = smooth_curve(measured_x[valid], measured_y[valid])
            ax.plot(smooth_x, smooth_y, linewidth=1.8, color=line_colors[arch], label=arch)
            ax.scatter(measured_x[valid], measured_y[valid], s=13, color=line_colors[arch], zorder=3)
        ax.set_title(technique_labels[technique])
        ax.grid(False)
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.suptitle('Mean across topologies, workloads, and repetitions', fontsize=11)
    fig.supxlabel('Simulation Time (min.)', y=0.02)
    fig.supylabel(ylabel, x=0.02)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=len(arch_order), frameon=False)
    fig.tight_layout(rect=(0.04, 0.04, 1, 0.93))
    fig.savefig(PLOTS_DIR / filename, bbox_inches='tight')
    plt.close(fig)
    display(IFrame(src=f'plots/{filename}', width='100%', height=430))
